In [1]:
#useful python libraries
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
#sklearn modules
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from xgboost import XGBClassifier

In [2]:
#load my data
df = pd.read_parquet("my_feature_space.parquet")
df

,objectId,finkclass,mean,weighted_mean,standard_deviation,median,amplitude,beyond_1_std,cusum,inter_percentile_range_10,...,magnitude_percentage_ratio_20_10,maximum_slope,median_absolute_deviation,median_buffer_range_percentage_10,percent_amplitude,mean_variance,anderson_darling_normal,chi2,skew,stetson_K
0,ZTF17aaaadkj,CataclyV*,17.222114,17.174713,0.237890,17.224249,1.171396,0.195322,0.108385,0.481224,...,0.676086,410.613047,0.126921,0.457310,1.825376,0.013813,13.875019,242.808733,-2.438629,0.585522
1,ZTF17aaaagyq,CataclyV*,16.735330,16.498882,0.637000,16.862818,1.910264,0.197590,0.152603,1.436352,...,0.429171,354.654346,0.234251,0.414458,2.552179,0.038063,43.600593,2694.289711,-1.706663,0.709195
2,ZTF17aaaaqna,Unknown,14.323769,14.321861,0.206714,14.224896,0.440336,0.169289,0.109419,0.441297,...,0.723493,95.102274,0.062597,0.344004,0.750938,0.014432,138.605800,238.412293,1.464617,0.796598
3,ZTF17aaaarmr,CataclyV*,16.282178,16.256730,0.240892,16.229107,1.577422,0.129736,0.238448,0.418315,...,0.507220,137.920000,0.074253,0.771527,2.849228,0.014795,61.877870,121.862796,5.046224,0.701406
4,ZTF17aaaazob,CataclyV*,17.864093,17.649710,0.403722,17.859806,2.242684,0.179581,0.082159,0.778325,...,0.627876,696.187235,0.198764,0.561076,3.199834,0.022600,49.381505,620.820245,-2.514259,0.488813
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2054,ZTF26aaajqnu,CataclyV*,16.513851,16.289990,0.722135,16.555887,3.018303,0.117801,0.068421,0.868633,...,0.642155,243.547828,0.209332,0.628272,3.193780,0.043729,30.237867,2277.639225,-1.465526,0.593412
2055,ZTF26aaaombw,Unknown,15.516807,15.516408,0.037666,15.522238,0.106918,0.317241,0.289659,0.100583,...,0.651676,83.947826,0.024834,0.230345,0.113693,0.002427,7.927470,12.038277,-0.567986,0.803067
2056,ZTF26aabsgfq,Unknown,14.029994,14.029941,0.029484,14.032178,0.103438,0.282857,0.198469,0.072323,...,0.627597,29.321250,0.017713,0.320000,0.111945,0.002101,1.771802,7.516016,-0.083897,0.769782
2057,ZTF26aaewgqp,Unknown,14.614267,14.612863,0.066856,14.611416,0.766450,0.051392,0.122444,0.084457,...,0.658268,181.159533,0.022746,0.966809,1.428257,0.004575,76.158691,14.791354,12.549345,0.479608


In [3]:
# 1 = CataclyV (including candidates)
# 0 = everything else

y_true = df['finkclass'].apply(
    lambda x: 1 if 'Cat' in str(x) else 0     #true class title
).values

In [4]:
# Load saved model
#bestmodel at first
best_model = joblib.load("cv_classifier_best_model.pkl")

In [5]:
#feature space
feature_columns = df.drop(columns=['objectId', 'finkclass']).columns #drop non_numeric columns

X = df[feature_columns].values #feature column

In [6]:
y_pred = best_model.predict(X) #predict model
print("Unique predictions:", np.unique(y_pred)) #cross check

Unique predictions: [0 1]


In [7]:
# Check class probabilities for the test set.
# predict_proba returns an array of shape (N_test, 2),
# where N_test is the number of test objects.
# Column 0 : probability of class 0 (negative, unknown)
# Column 1 : probability of class +1 (positive, cvs)
probs = best_model.predict_proba(X)
print('probs',probs)

probs [[0.02976841 0.9702316 ]
 [0.7804266  0.2195734 ]
 [0.9801404  0.01985963]
 ...
 [0.87668353 0.12331649]
 [0.9762438  0.02375618]
 [0.9351252  0.06487483]]


In [8]:
# CV probability
cv_prob = probs[:,1]

# create dataframe for attaching objectid
prob_df = pd.DataFrame({
    "objectId": df["objectId"],
    "cv_probability": cv_prob
})
#check output
print(prob_df.head())

       objectId  cv_probability
0  ZTF17aaaadkj        0.970232
1  ZTF17aaaagyq        0.219573
2  ZTF17aaaaqna        0.019860
3  ZTF17aaaarmr        0.030531
4  ZTF17aaaazob        0.805806


In [9]:
# CV probability with finkclass_column_atached
cv_prob = probs[:,1]

# create dataframe with required columns
prob_df = pd.DataFrame({
    "objectId": df["objectId"],
    "finkclass": df["finkclass"],
    "cv_probability": cv_prob
})

# check
print(prob_df.head())

       objectId  finkclass  cv_probability
0  ZTF17aaaadkj  CataclyV*        0.970232
1  ZTF17aaaagyq  CataclyV*        0.219573
2  ZTF17aaaaqna    Unknown        0.019860
3  ZTF17aaaarmr  CataclyV*        0.030531
4  ZTF17aaaazob  CataclyV*        0.805806


In [10]:
#saving into file
prob_df.to_csv("best_model_prediction.csv", index=False)

In [11]:
prob_df

,objectId,finkclass,cv_probability
0,ZTF17aaaadkj,CataclyV*,0.970232
1,ZTF17aaaagyq,CataclyV*,0.219573
2,ZTF17aaaaqna,Unknown,0.019860
3,ZTF17aaaarmr,CataclyV*,0.030531
4,ZTF17aaaazob,CataclyV*,0.805806
...,...,...,...
2054,ZTF26aaajqnu,CataclyV*,0.984470
2055,ZTF26aaaombw,Unknown,0.070568
2056,ZTF26aabsgfq,Unknown,0.123316
2057,ZTF26aaewgqp,Unknown,0.023756


In [12]:
#probablity distribution checking
high_prob = prob_df[prob_df["cv_probability"] > 0.5]
print("Number of objects with CV probability > 0.5:", len(high_prob))

Number of objects with CV probability > 0.5: 907


In [13]:
#saving into parquet
high_prob.to_parquet("best_model_prediction_cv_candidates_above_0.5.parquet", index=False)

In [14]:
high_prob

,objectId,finkclass,cv_probability
0,ZTF17aaaadkj,CataclyV*,0.970232
4,ZTF17aaaazob,CataclyV*,0.805806
6,ZTF17aaabavb,CataclyV*,0.931277
7,ZTF17aaabfay,CataclyV*,0.840810
8,ZTF17aaabfbg,CataclyV*,0.844816
...,...,...,...
2044,ZTF25abuhmys,Unknown,0.939034
2045,ZTF25abungcd,CataclyV*,0.996571
2047,ZTF25acfrkfu,Unknown,0.658873
2051,ZTF26aaaelef,CataclyV*,0.866908


In [15]:
#now for previously saved model
# Load saved model
model = joblib.load("cv_classifier_xgb_boost_biggie_set.pkl")

In [16]:
y_pred_1 = model.predict(X) #predict model

In [17]:
# Check class probabilities for the test set.
# predict_proba returns an array of shape (N_test, 2),
# where N_test is the number of test objects.
# Column 0 : probability of class 0 (negative, unknown)
# Column 1 : probability of class +1 (positive, cvs)
probs_1 = model.predict_proba(X)
print('probs_1',probs_1)

probs_1 [[0.0079872  0.9920128 ]
 [0.46320283 0.53679717]
 [0.9308319  0.06916811]
 ...
 [0.86918795 0.13081206]
 [0.9826581  0.01734191]
 [0.9857574  0.01424257]]


In [18]:
# CV probability
cv_prob_1 = probs_1[:,1]

# create dataframe for attaching objectid
prob_df_1 = pd.DataFrame({
    "objectId": df["objectId"],
    "finkclass": df["finkclass"],
    "cv_probability": cv_prob_1
})
#check output
print(prob_df_1.head())

       objectId  finkclass  cv_probability
0  ZTF17aaaadkj  CataclyV*        0.992013
1  ZTF17aaaagyq  CataclyV*        0.536797
2  ZTF17aaaaqna    Unknown        0.069168
3  ZTF17aaaarmr  CataclyV*        0.111012
4  ZTF17aaaazob  CataclyV*        0.634392


In [19]:
#saving into parquet
prob_df_1.to_csv("class prediction for previousy saved model.parquet", index=False)

In [20]:
prob_df_1

,objectId,finkclass,cv_probability
0,ZTF17aaaadkj,CataclyV*,0.992013
1,ZTF17aaaagyq,CataclyV*,0.536797
2,ZTF17aaaaqna,Unknown,0.069168
3,ZTF17aaaarmr,CataclyV*,0.111012
4,ZTF17aaaazob,CataclyV*,0.634392
...,...,...,...
2054,ZTF26aaajqnu,CataclyV*,0.990807
2055,ZTF26aaaombw,Unknown,0.027301
2056,ZTF26aabsgfq,Unknown,0.130812
2057,ZTF26aaewgqp,Unknown,0.017342


In [21]:
#probablity distribution checking
high_prob_1 = prob_df_1[prob_df_1["cv_probability"] > 0.5]
print("Number of objects with CV probability > 0.5:", len(high_prob_1))

Number of objects with CV probability > 0.5: 925


In [22]:
#saving into parquet
high_prob_1.to_parquet("cv_candidates_above_0.5.parquet_for_previously_saved_model.parquet", index=False)

In [23]:
high_prob_1

,objectId,finkclass,cv_probability
0,ZTF17aaaadkj,CataclyV*,0.992013
1,ZTF17aaaagyq,CataclyV*,0.536797
4,ZTF17aaaazob,CataclyV*,0.634392
6,ZTF17aaabavb,CataclyV*,0.988868
7,ZTF17aaabfay,CataclyV*,0.719409
...,...,...,...
2044,ZTF25abuhmys,Unknown,0.599309
2045,ZTF25abungcd,CataclyV*,0.997949
2047,ZTF25acfrkfu,Unknown,0.552462
2051,ZTF26aaaelef,CataclyV*,0.922728


In [24]:
#compare b/w them
y_pred_1 = best_model.predict(X)
y_pred_2 = model.predict(X)

In [25]:
#accuracy score
print("best_model:", accuracy_score(y_true, y_pred_1))
print("model:", accuracy_score(y_true, y_pred_2))

best_model: 0.740165128703254
model: 0.7489072365225837


In [26]:
#find the mismatch
#load high probability files
high_prob_best = pd.read_parquet(
    "best_model_prediction_cv_candidates_above_0.5.parquet"
)

high_prob_old = pd.read_parquet(
    "cv_candidates_above_0.5.parquet_for_previously_saved_model.parquet"
)

In [27]:
#mismatches
cv_best = set(high_prob_best["objectId"])
cv_old = set(high_prob_old["objectId"])

only_in_best = cv_best - cv_old
only_in_old = cv_old - cv_best

print("Only in best model:", len(only_in_best))
print("Only in old model:", len(only_in_old))

Only in best model: 95
Only in old model: 113


In [28]:
#extra objects of new model
extra_best = high_prob_best[
    high_prob_best["objectId"].isin(only_in_best)
]
#extra objects of old model
extra_old = high_prob_old[
    high_prob_old["objectId"].isin(only_in_old)
]
#print
print(extra_best)
print(extra_old)

         objectId  finkclass  cv_probability
25   ZTF17aaairkd  CataclyV*        0.516564
47   ZTF17aaaruaj  CataclyV*        0.769591
49   ZTF17aaasaat  CataclyV*        0.818963
51   ZTF17aaasenq  CataclyV*        0.678569
57   ZTF17aaawcrh  CataclyV*        0.640926
..            ...        ...             ...
838  ZTF20acywmfj  CataclyV*        0.633712
848  ZTF21aagorst    Unknown        0.618482
849  ZTF21aahaola    Unknown        0.583795
865  ZTF22aaeydmw    Unknown        0.564999
885  ZTF23abiplzg    Unknown        0.527050

[95 rows x 3 columns]
         objectId  finkclass  cv_probability
1    ZTF17aaaagyq  CataclyV*        0.536797
9    ZTF17aaacqrw  CataclyV*        0.696722
13   ZTF17aaaentq  CataclyV*        0.767871
15   ZTF17aaaewmi  CataclyV*        0.640854
19   ZTF17aaagtoh    Unknown        0.673578
..            ...        ...             ...
892  ZTF23aabquaf    Unknown        0.767827
897  ZTF23aaowhda    Unknown        0.620049
901  ZTF23abgucqy    Unknown    

In [29]:
#coomon elements
common_ids = cv_best.intersection(cv_old)

print("Common CV candidates:", len(common_ids))

Common CV candidates: 812


In [30]:
#change in probability
common_best = high_prob_best[
    high_prob_best["objectId"].isin(common_ids)
]

common_old = high_prob_old[
    high_prob_old["objectId"].isin(common_ids)
]

merged = common_best.merge(
    common_old,
    on="objectId",
    suffixes=("_best", "_old")
)

merged["prob_diff"] = abs(
    merged["cv_probability_best"]
    - merged["cv_probability_old"]
)

In [31]:
#show
merged.sort_values(
    "prob_diff",
    ascending=True
).head(20)

,objectId,finkclass_best,cv_probability_best,finkclass_old,cv_probability_old,prob_diff
133,ZTF18aaaatwt,CataclyV*,0.886893,CataclyV*,0.887247,0.000354
782,ZTF22absafly,Unknown,0.965441,Unknown,0.964855,0.000586
11,ZTF17aaaeslm,CataclyV*,0.989034,CataclyV*,0.989662,0.000628
356,ZTF18abpokwo,CataclyV*,0.997499,CataclyV*,0.996790,0.000709
613,ZTF19aaaoiql,CataclyV*,0.978287,CataclyV*,0.979001,0.000714
751,ZTF20adcaqij,CataclyV*,0.973742,CataclyV*,0.974487,0.000745
551,ZTF18adalbvx,CataclyV*,0.994021,CataclyV*,0.994772,0.000751
203,ZTF18aabqvcj,CataclyV*,0.745052,CataclyV*,0.745936,0.000884
666,ZTF19aapwyle,CataclyV*,0.996319,CataclyV*,0.997224,0.000905
799,ZTF24abyucsd,CataclyV*,0.997600,CataclyV*,0.998605,0.001005
